In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# Output: Using device: cuda  (if GPU available)
# Output: Using device: cpu   (if no GPU)


Using device: cuda


In [4]:
print("\nLoading MRPC dataset...")
dataset = load_dataset("glue", "mrpc")

print(dataset)
sample = dataset['train'][0]
print("\nSample Example:")
print(f"  Sentence 1 : {sample['sentence1']}")
print(f"  Sentence 2 : {sample['sentence2']}")
print(f"  Label      : {sample['label']}  (1=paraphrase, 0=not)")


Loading MRPC dataset...
DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

Sample Example:
  Sentence 1 : Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .
  Sentence 2 : Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .
  Label      : 1  (1=paraphrase, 0=not)


In [5]:
print("\nLoading tokenizer...")
MODEL_NAME = "bert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Vocabulary size : {tokenizer.vocab_size}")
print(f"Max length      : {tokenizer.model_max_length}")



Loading tokenizer...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocabulary size : 30522
Max length      : 512


In [7]:
print("\n--- Tokenization Demo ---")

s1 = "He said the food was delicious."
s2 = "He mentioned the meal tasted great."

# Tokenize a single sentence
tokens = tokenizer.tokenize(s1)
print(f"\nTokens       : {tokens}")

# Encode single sentence
ids = tokenizer.encode(s1)
print(f"Encoded IDs  : {ids}")
# [101, ...tokens..., 102]
#  ^CLS              ^SEP

# Encode a PAIR of sentences (for MRPC)
pair_encoding = tokenizer(
    s1,
    s2,
    add_special_tokens=True,
    max_length=128,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

print(f"\nInput IDs shape      : {pair_encoding['input_ids'].shape}")
print(f"Attention mask shape : {pair_encoding['attention_mask'].shape}")
print(f"Token type IDs shape : {pair_encoding['token_type_ids'].shape}")

# Decode back to see what was tokenized
decoded = tokenizer.decode(
    pair_encoding['input_ids'][0],
    skip_special_tokens=False
)
print(f"\nDecoded tokens: {decoded[:120]}...")
# [CLS] sentence1 tokens [SEP] sentence2 tokens [SEP]

print(f"\nToken Type IDs (first 20): {pair_encoding['token_type_ids'][0][:20].tolist()}")
# 0 = Sentence A tokens
# 1 = Sentence B tokens




--- Tokenization Demo ---

Tokens       : ['he', 'said', 'the', 'food', 'was', 'delicious', '.']
Encoded IDs  : [101, 2002, 2056, 1996, 2833, 2001, 12090, 1012, 102]

Input IDs shape      : torch.Size([1, 128])
Attention mask shape : torch.Size([1, 128])
Token type IDs shape : torch.Size([1, 128])

Decoded tokens: [CLS] he said the food was delicious. [SEP] he mentioned the meal tasted great. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD...

Token Type IDs (first 20): [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]


In [8]:
print("\nTokenizing dataset...")

def tokenize_fn(examples):
    return tokenizer(
        examples['sentence1'],
        examples['sentence2'],
        padding    = 'max_length',
        truncation = True,
        max_length = 128
    )

# Apply tokenization to all splits
tokenized_dataset = dataset.map(tokenize_fn, batched=True)

# Set format for PyTorch
tokenized_dataset.set_format(
    type    = 'torch',
    columns = ['input_ids', 'attention_mask', 'token_type_ids', 'label']
)

print("Tokenized dataset ready!")
print(f"Train columns: {tokenized_dataset['train'].column_names}")




Tokenizing dataset...


Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

Tokenized dataset ready!
Train columns: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask']


In [9]:
print("\nLoading BERT model...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = 2   # Binary: paraphrase or not
)
model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable:,}")




Loading BERT model...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters    : 109,483,778
Trainable parameters: 109,483,778


In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average='binary')
    return {"accuracy": acc, "f1": f1}


In [13]:
training_args = TrainingArguments(
    output_dir              = "./bert_mrpc_output",
    num_train_epochs        = 3,          # Train for 3 epochs
    per_device_train_batch_size = 16,     # 16 samples per batch
    per_device_eval_batch_size  = 32,
    learning_rate           = 2e-5,       # BERT standard learning rate
    weight_decay            = 0.01,       # Regularization
    eval_strategy     = "epoch",   # Evaluate after each epoch
    save_strategy           = "epoch",
    load_best_model_at_end  = True,
    metric_for_best_model   = "f1",
    logging_steps           = 50,
    report_to               = "none"      # Disable wandb/tensorboard
)


In [15]:
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized_dataset['train'],
    eval_dataset    = tokenized_dataset['validation'],
    processing_class       = tokenizer,
    compute_metrics = compute_metrics
)

print("\nStarting training...")
print("(This takes ~3-5 mins on Colab GPU)\n")
trainer.train()



Starting training...
(This takes ~3-5 mins on Colab GPU)



Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.516319,0.409852,0.821078,0.876897
2,0.339648,0.344730,0.850490,0.893170
3,0.154483,0.407041,0.852941,0.897260


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=690, training_loss=0.34580912313599516, metrics={'train_runtime': 399.0406, 'train_samples_per_second': 27.576, 'train_steps_per_second': 1.729, 'total_flos': 723818513295360.0, 'train_loss': 0.34580912313599516, 'epoch': 3.0})

In [16]:
print("\nEvaluating...")
results = trainer.evaluate()

print(f"\nValidation Results:")
print(f"  Accuracy : {results['eval_accuracy']:.4f}")
print(f"  F1 Score : {results['eval_f1']:.4f}")
print(f"  Loss     : {results['eval_loss']:.4f}")
# Expected: Accuracy ~0.86, F1 ~0.89



Evaluating...



Validation Results:
  Accuracy : 0.8529
  F1 Score : 0.8973
  Loss     : 0.4070


In [17]:
print("\n--- Making Predictions ---")

def predict_similarity(sentence1, sentence2):
    """Predict if two sentences are paraphrases."""
    inputs = tokenizer(
        sentence1,
        sentence2,
        return_tensors  = 'pt',
        max_length      = 128,
        padding         = 'max_length',
        truncation      = True
    ).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits  = outputs.logits

    probabilities = torch.softmax(logits, dim=-1)
    prediction    = torch.argmax(logits, dim=-1).item()

    label = "PARAPHRASE ✓" if prediction == 1 else "NOT Paraphrase ✗"
    conf  = probabilities[0][prediction].item()

    print(f"\nSentence 1 : {sentence1}")
    print(f"Sentence 2 : {sentence2}")
    print(f"Prediction : {label}")
    print(f"Confidence : {conf:.2%}")
    return prediction


# Test with examples
predict_similarity(
    "The stock prices fell sharply yesterday.",
    "Share values dropped significantly the previous day."
)

predict_similarity(
    "She loves playing the piano.",
    "The weather forecast predicts heavy rain."
)

predict_similarity(
    "The company announced record profits.",
    "The firm reported its highest ever earnings."
)



--- Making Predictions ---

Sentence 1 : The stock prices fell sharply yesterday.
Sentence 2 : Share values dropped significantly the previous day.
Prediction : PARAPHRASE ✓
Confidence : 91.78%

Sentence 1 : She loves playing the piano.
Sentence 2 : The weather forecast predicts heavy rain.
Prediction : NOT Paraphrase ✗
Confidence : 96.09%

Sentence 1 : The company announced record profits.
Sentence 2 : The firm reported its highest ever earnings.
Prediction : NOT Paraphrase ✗
Confidence : 50.77%


0

In [18]:
print("\nSaving model...")
model.save_pretrained("./my_bert_mrpc")
tokenizer.save_pretrained("./my_bert_mrpc")
print("Model saved to ./my_bert_mrpc")


# Load and Reuse Saved Model
print("\nLoading saved model...")
loaded_model     = AutoModelForSequenceClassification.from_pretrained("./my_bert_mrpc")
loaded_tokenizer = AutoTokenizer.from_pretrained("./my_bert_mrpc")
print("Model loaded successfully!")



Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./my_bert_mrpc

Loading saved model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully!
